# Mouse Skin Atlas - Downstream Analysis

This notebook performs comprehensive downstream analysis on the integrated skin atlas to understand:
- Normal vs pathological wound healing
- Temporal dynamics of healing
- Cell type composition changes across conditions
- Key genes and pathways involved in healing

**Prerequisites**: Run `01_scvi_integration.ipynb` first to generate the integrated atlas.

In [1]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple
import anndata as ad
from scipy import stats
from statsmodels.stats.multitest import multipletests

# scVI imports
import scvi
from scvi.model import SCVI

# Settings
warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

# Paths
OUTPUT_DIR = Path("output")
ANALYSIS_DIR = OUTPUT_DIR / "analysis"
ANALYSIS_DIR.mkdir(exist_ok=True)

print(f"scanpy version: {sc.__version__}")
print(f"scvi version: {scvi.__version__}")

scanpy version: 1.11.5
scvi version: 1.4.1


## 1. Load Integrated Atlas

In [2]:
# Load integrated atlas
adata = sc.read_h5ad(OUTPUT_DIR / 'integrated_atlas_scvi.h5ad')
print(f"Loaded atlas: {adata.shape}")
print(f"Studies: {adata.obs['study'].unique().tolist()}")
print(f"Cell types: {adata.obs['cell_type'].nunique()}")

Loaded atlas: (186323, 3000)
Studies: ['BurnSham', 'LPCAT3', 'XXO_CKO']
Cell types: 23


In [3]:
# Load scVI model for DE analysis
model = SCVI.load(OUTPUT_DIR / 'scvi_model', adata=adata)
print("scVI model loaded")

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


INFO     File output/scvi_model/model.pt already downloaded                                                        
scVI model loaded


In [4]:
# Overview of the dataset
print("\n=== Dataset Overview ===")
print(f"Total cells: {adata.n_obs:,}")
print(f"Total genes: {adata.n_vars:,}")

print("\nCells per study:")
display(adata.obs['study'].value_counts())

print("\nCondition/Type distribution:")
display(adata.obs['Type'].value_counts())


=== Dataset Overview ===
Total cells: 186,323
Total genes: 3,000

Cells per study:


study
XXO_CKO     98209
BurnSham    57293
LPCAT3      30821
Name: count, dtype: int64


Condition/Type distribution:


Type
Sham             32441
XXT              26349
Burn             24852
XYT              24468
XYO              19393
XXO              19247
LPCAT3 KO GAS    12941
WT (NS)           9546
LPCAT3 WT GAS     8334
WT                5395
CKO               3357
Name: count, dtype: int64

## 2. Cell Type Annotation and Harmonization

Since datasets may have different cell type annotations, we'll harmonize them and annotate unlabeled cells.

In [5]:
# Current cell type distribution by study
ct_by_study = pd.crosstab(adata.obs['cell_type'], adata.obs['study'])
print("Cell types by study:")
display(ct_by_study)

Cell types by study:


study,BurnSham,LPCAT3,XXO_CKO
cell_type,,,
Ad,108,0,0
Adipocytes,0,147,0
DC/Mo,2263,0,0
Dendritic Cells,0,1001,0
EC,2031,0,0
Endothelial Cells,0,709,0
Fib,11299,0,0
Fibroblasts,0,417,0
Inflammatory Fibroblasts,0,989,0


In [ ]:
# Define canonical markers for skin cell types
CELL_TYPE_MARKERS = {
    # Epithelial
    'Keratinocytes': ['Krt14', 'Krt5', 'Krt1', 'Krt10', 'Krt17'],
    'Basal Keratinocytes': ['Krt14', 'Krt5', 'Itga6', 'Itgb1'],
    'Suprabasal Keratinocytes': ['Krt1', 'Krt10', 'Ivl', 'Lor'],
    
    # Mesenchymal
    'Fibroblasts': ['Col1a1', 'Col1a2', 'Dcn', 'Lum', 'Pdgfra'],
    'Myofibroblasts': ['Acta2', 'Tagln', 'Col1a1', 'Postn'],
    
    # Endothelial & Lymphatic
    'Endothelial': ['Pecam1', 'Cdh5', 'Vwf', 'Kdr', 'Emcn'],
    'Lymphatic Endothelial': ['Prox1', 'Lyve1', 'Pdpn', 'Flt4'],
    
    # Immune - Myeloid
    'Macrophages': ['Cd68', 'Adgre1', 'Csf1r', 'Mrc1', 'Cd163'],
    'M1-like Macrophages': ['Nos2', 'Tnf', 'Il1b', 'Cd86'],
    'M2-like Macrophages': ['Arg1', 'Mrc1', 'Cd163', 'Chil3', "Trem2", 'Retnla'],
    'Monocytes': ['Ly6c2', 'Ccr2', 'Cd14'],
    'Neutrophils': ['S100a8', 'S100a9', 'Ly6g', 'Cxcr2', 'Mmp9'],
    'Dendritic cells': ['Itgax', 'Cd74', 'H2-Aa', 'Flt3', 'Xcr1'],
    'Langerhans cells': ['Cd207', 'Epcam', 'Cd74'],
    'Mast cells': ['Kit', 'Cpa3', 'Tpsb2', 'Mcpt4'],
    
    # Immune - Lymphoid
    'T cells': ['Cd3e', 'Trac', 'Cd4', 'Cd3d', 'Il7r','Cd8a', 'Foxp3', 'Il2ra', 'Ctla4'],
    'NK cells': ['Ncr1', 'Nkg7', 'Klrb1c', 'Gzma'],
    'ILCs': ['Il7r', 'Id2', 'Gata3', 'Rorc'],
    
    # Other
    'Smooth Muscle': ['Acta2', 'Myh11', 'Tagln', 'Des', 'Cnn1'],
    'Fascia cells': ['Mfap5', 'Wnt2', 'Creb5', 'Col14a1', "Tnnc2", 'Tmeff2'],
    'Panniculus Carnousus': ['Pax7', 'Myh2', 'Myog', 'Myf5'],
    'Pericytes': ['Pdgfrb', 'Rgs5', 'Notch3', 'Acta2'],
    'Schwann cells': ['Mbp', 'Mpz', 'Sox10', 'Plp1'],
    'Melanocytes': ['Dct', 'Tyrp1', 'Pmel', 'Mitf'],
    'Adipocytes': ['Adipoq', 'Lep', 'Pparg', 'Fabp4'],
}

# Filter to available markers
available_markers = {}
for cell_type, markers in CELL_TYPE_MARKERS.items():
    present = [m for m in markers if m in adata.var_names]
    if len(present) >= 2:  # At least 2 markers present
        available_markers[cell_type] = present

print(f"Cell types with markers available: {len(available_markers)}")

In [ ]:
# Compute marker scores for each cell type
def compute_marker_scores(adata: ad.AnnData, marker_dict: Dict[str, List[str]]) -> pd.DataFrame:
    """
    Compute average expression score for each cell type's markers.
    """
    scores = {}
    
    for cell_type, markers in marker_dict.items():
        # Get expression for available markers
        marker_expr = adata[:, markers].X
        if hasattr(marker_expr, 'toarray'):
            marker_expr = marker_expr.toarray()
        
        # Mean expression across markers
        scores[cell_type] = marker_expr.mean(axis=1)
    
    return pd.DataFrame(scores, index=adata.obs_names)


# Compute scores
marker_scores = compute_marker_scores(adata, available_markers)

# Add to adata.obs
for ct in marker_scores.columns:
    adata.obs[f'score_{ct}'] = marker_scores[ct].values

In [ ]:
# Annotate Unknown cells based on highest marker score
def annotate_unknown_cells(
    adata: ad.AnnData,
    marker_scores: pd.DataFrame,
    cell_type_col: str = 'cell_type',
    min_score_threshold: float = 0.5,
) -> pd.Series:
    """
    Annotate cells with Unknown cell type based on marker scores.
    """
    # Convert to string to avoid categorical issues
    new_annotations = adata.obs[cell_type_col].astype(str).copy()
    
    # Find Unknown cells
    unknown_mask = new_annotations == 'Unknown'
    n_unknown = unknown_mask.sum()
    
    if n_unknown > 0:
        print(f"Annotating {n_unknown} Unknown cells...")
        
        # Get best matching cell type for unknown cells
        unknown_scores = marker_scores.loc[unknown_mask]
        best_types = unknown_scores.idxmax(axis=1)
        best_scores = unknown_scores.max(axis=1)
        
        # Only assign if score is above threshold
        confident = best_scores >= min_score_threshold
        
        # Update annotations for confident cells
        confident_indices = unknown_mask[unknown_mask].index[confident.values]
        new_annotations.loc[confident_indices] = best_types[confident].values
        
        print(f"  Annotated {confident.sum()} cells with confidence >= {min_score_threshold}")
        print(f"  Remaining Unknown: {(new_annotations == 'Unknown').sum()}")
    else:
        print("No Unknown cells to annotate")
    
    return new_annotations


# Annotate unknown cells
adata.obs['cell_type_harmonized'] = annotate_unknown_cells(adata, marker_scores)

In [ ]:
# Visualize cell types
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc.pl.umap(adata, color='cell_type', ax=axes[0], show=False, title='Original Cell Types', legend_loc='right margin')
sc.pl.umap(adata, color='cell_type_harmonized', ax=axes[1], show=False, title='Harmonized Cell Types', legend_loc='right margin')

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'cell_type_harmonization.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Marker gene heatmap by cluster
sc.pl.dotplot(
    adata,
    var_names=available_markers,
    groupby='leiden_scvi',
    standard_scale='var',
    dendrogram=True,
    figsize=(20, 8),
)
plt.savefig(ANALYSIS_DIR / 'marker_dotplot_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Cell Type Composition Analysis

Compare cell type proportions across conditions to understand how healing affects cellular composition.

In [ ]:
def compute_cell_type_proportions(
    adata: ad.AnnData,
    cell_type_col: str = 'cell_type_harmonized',
    groupby: str = 'Sample',
    condition_col: str = 'Type',
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute cell type proportions per sample and condition.
    
    Returns
    -------
    proportions : DataFrame
        Cell type proportions per sample
    sample_info : DataFrame
        Sample metadata including condition
    """
    # Count cells per sample and cell type
    counts = pd.crosstab(adata.obs[groupby], adata.obs[cell_type_col])
    
    # Convert to proportions
    proportions = counts.div(counts.sum(axis=1), axis=0)
    
    # Get sample metadata
    sample_info = adata.obs.groupby(groupby).agg({
        condition_col: 'first',
        'study': 'first',
    }).reset_index()
    sample_info = sample_info.set_index(groupby)
    
    return proportions, sample_info


# Compute proportions
proportions, sample_info = compute_cell_type_proportions(adata)
print(f"Samples: {len(proportions)}")
print(f"Cell types: {len(proportions.columns)}")

In [ ]:
# Stacked bar plot of cell type proportions by condition
def plot_composition_by_condition(
    proportions: pd.DataFrame,
    sample_info: pd.DataFrame,
    condition_col: str = 'Type',
) -> plt.Figure:
    """
    Create stacked bar plot of cell type proportions by condition.
    """
    # Add condition to proportions
    prop_with_condition = proportions.join(sample_info[[condition_col]])
    
    # Average proportions by condition
    mean_props = prop_with_condition.groupby(condition_col).mean()
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    mean_props.plot(kind='bar', stacked=True, ax=ax, colormap='tab20')
    
    ax.set_xlabel('Condition')
    ax.set_ylabel('Cell Type Proportion')
    ax.set_title('Cell Type Composition by Condition')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Cell Type')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig


fig = plot_composition_by_condition(proportions, sample_info)
plt.savefig(ANALYSIS_DIR / 'cell_composition_by_condition.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Statistical comparison of cell type proportions
def compare_proportions(
    proportions: pd.DataFrame,
    sample_info: pd.DataFrame,
    condition_col: str = 'Type',
    group1: str = None,
    group2: str = None,
) -> pd.DataFrame:
    """
    Compare cell type proportions between two conditions using Mann-Whitney U test.
    """
    conditions = sample_info[condition_col].unique()
    
    if group1 is None or group2 is None:
        if len(conditions) >= 2:
            group1, group2 = conditions[:2]
        else:
            return pd.DataFrame()
    
    # Get samples for each group
    samples_g1 = sample_info[sample_info[condition_col] == group1].index
    samples_g2 = sample_info[sample_info[condition_col] == group2].index
    
    results = []
    for cell_type in proportions.columns:
        vals_g1 = proportions.loc[proportions.index.isin(samples_g1), cell_type]
        vals_g2 = proportions.loc[proportions.index.isin(samples_g2), cell_type]
        
        if len(vals_g1) > 1 and len(vals_g2) > 1:
            stat, pval = stats.mannwhitneyu(vals_g1, vals_g2, alternative='two-sided')
            
            results.append({
                'cell_type': cell_type,
                f'mean_{group1}': vals_g1.mean(),
                f'mean_{group2}': vals_g2.mean(),
                'log2_fold_change': np.log2((vals_g2.mean() + 1e-6) / (vals_g1.mean() + 1e-6)),
                'pvalue': pval,
            })
    
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df['pvalue_adj'] = multipletests(results_df['pvalue'], method='fdr_bh')[1]
        results_df = results_df.sort_values('pvalue')
    
    return results_df


# Compare Burn vs Sham (if available)
if 'Burn' in sample_info['Type'].values and 'Sham' in sample_info['Type'].values:
    burn_vs_sham = compare_proportions(proportions, sample_info, group1='Sham', group2='Burn')
    print("Burn vs Sham - Cell Type Proportion Changes:")
    display(burn_vs_sham.head(10))
    burn_vs_sham.to_csv(ANALYSIS_DIR / 'burn_vs_sham_composition.csv', index=False)

In [ ]:
# Visualize proportion changes
def plot_proportion_comparison(
    proportions: pd.DataFrame,
    sample_info: pd.DataFrame,
    comparison_df: pd.DataFrame,
    top_n: int = 10,
    condition_col: str = 'Type',
) -> plt.Figure:
    """
    Box plots of cell type proportions for top differentially abundant cell types.
    """
    if comparison_df is None or len(comparison_df) == 0:
        return None
    
    # Get top cell types by significance
    top_types = comparison_df.head(top_n)['cell_type'].tolist()
    
    # Prepare data for plotting
    plot_data = proportions[top_types].copy()
    plot_data[condition_col] = sample_info[condition_col]
    plot_data_melted = plot_data.melt(
        id_vars=[condition_col],
        var_name='Cell Type',
        value_name='Proportion'
    )
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.boxplot(
        data=plot_data_melted,
        x='Cell Type',
        y='Proportion',
        hue=condition_col,
        ax=ax,
    )
    
    ax.set_xlabel('')
    ax.set_ylabel('Proportion')
    ax.set_title('Cell Type Proportions - Top Differentially Abundant')
    ax.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    return fig


if 'burn_vs_sham' in dir() and len(burn_vs_sham) > 0:
    fig = plot_proportion_comparison(proportions, sample_info, burn_vs_sham)
    if fig:
        plt.savefig(ANALYSIS_DIR / 'burn_vs_sham_boxplot.png', dpi=150, bbox_inches='tight')
        plt.show()

## 4. Differential Expression Analysis

Use scVI's built-in differential expression to identify genes associated with healing and pathology.

In [ ]:
def run_scvi_de(
    model: SCVI,
    adata: ad.AnnData,
    groupby: str,
    group1: str,
    group2: str,
    n_samples: int = 5000,
) -> pd.DataFrame:
    """
    Run differential expression using scVI.
    """
    # Create masks
    idx1 = adata.obs[groupby] == group1
    idx2 = adata.obs[groupby] == group2
    
    print(f"Comparing {group1} ({idx1.sum()} cells) vs {group2} ({idx2.sum()} cells)")
    
    # Run DE
    de_results = model.differential_expression(
        idx1=idx1.values,
        idx2=idx2.values,
        n_samples=n_samples,
    )
    
    return de_results


# Global DE: Burn vs Sham (if available)
if 'Burn' in adata.obs['Type'].values and 'Sham' in adata.obs['Type'].values:
    de_burn_sham = run_scvi_de(model, adata, 'Type', 'Sham', 'Burn')
    
    # Filter significant genes
    sig_genes = de_burn_sham[
        (de_burn_sham['is_de_fdr_0.05'] == True) &
        (abs(de_burn_sham['lfc_mean']) > 0.5)
    ].sort_values('lfc_mean', ascending=False)
    
    print(f"\nSignificant DE genes (FDR < 0.05, |LFC| > 0.5): {len(sig_genes)}")
    print("\nTop upregulated in Burn:")
    display(sig_genes.head(10))
    print("\nTop downregulated in Burn:")
    display(sig_genes.tail(10))
    
    de_burn_sham.to_csv(ANALYSIS_DIR / 'de_burn_vs_sham.csv')

In [ ]:
# Cell type-specific DE
def run_celltype_specific_de(
    model: SCVI,
    adata: ad.AnnData,
    cell_type_col: str,
    condition_col: str,
    group1: str,
    group2: str,
    min_cells: int = 50,
) -> Dict[str, pd.DataFrame]:
    """
    Run DE analysis for each cell type separately.
    """
    results = {}
    cell_types = adata.obs[cell_type_col].unique()
    
    for ct in cell_types:
        ct_mask = adata.obs[cell_type_col] == ct
        
        # Check if enough cells in both conditions
        g1_mask = ct_mask & (adata.obs[condition_col] == group1)
        g2_mask = ct_mask & (adata.obs[condition_col] == group2)
        
        if g1_mask.sum() >= min_cells and g2_mask.sum() >= min_cells:
            print(f"\n{ct}: {g1_mask.sum()} {group1} vs {g2_mask.sum()} {group2}")
            
            try:
                de = model.differential_expression(
                    idx1=g1_mask.values,
                    idx2=g2_mask.values,
                )
                de['cell_type'] = ct
                results[ct] = de
            except Exception as e:
                print(f"  Error: {e}")
        else:
            print(f"Skipping {ct}: insufficient cells")
    
    return results


# Run cell type-specific DE for Burn vs Sham
if 'Burn' in adata.obs['Type'].values and 'Sham' in adata.obs['Type'].values:
    celltype_de = run_celltype_specific_de(
        model, adata,
        cell_type_col='cell_type_harmonized',
        condition_col='Type',
        group1='Sham',
        group2='Burn',
    )
    
    # Save results
    for ct, de in celltype_de.items():
        safe_name = ct.replace(' ', '_').replace('/', '_')
        de.to_csv(ANALYSIS_DIR / f'de_burn_vs_sham_{safe_name}.csv')

In [ ]:
# Volcano plot function
def plot_volcano(
    de_results: pd.DataFrame,
    lfc_col: str = 'lfc_mean',
    pval_col: str = 'proba_de',
    lfc_threshold: float = 0.5,
    pval_threshold: float = 0.05,
    title: str = 'Volcano Plot',
    top_n_labels: int = 10,
) -> plt.Figure:
    """
    Create volcano plot from DE results.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate -log10(1-proba_de) for visualization
    de_results = de_results.copy()
    de_results['neg_log10_pval'] = -np.log10(1 - de_results[pval_col] + 1e-10)
    
    # Color points
    colors = []
    for _, row in de_results.iterrows():
        if row[pval_col] > (1 - pval_threshold) and abs(row[lfc_col]) > lfc_threshold:
            if row[lfc_col] > 0:
                colors.append('red')
            else:
                colors.append('blue')
        else:
            colors.append('gray')
    
    ax.scatter(
        de_results[lfc_col],
        de_results['neg_log10_pval'],
        c=colors,
        alpha=0.5,
        s=10,
    )
    
    # Add threshold lines
    ax.axhline(-np.log10(pval_threshold), color='gray', linestyle='--', alpha=0.5)
    ax.axvline(lfc_threshold, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(-lfc_threshold, color='gray', linestyle='--', alpha=0.5)
    
    # Label top genes
    sig_up = de_results[(de_results[pval_col] > 0.95) & (de_results[lfc_col] > lfc_threshold)].nlargest(top_n_labels, lfc_col)
    sig_down = de_results[(de_results[pval_col] > 0.95) & (de_results[lfc_col] < -lfc_threshold)].nsmallest(top_n_labels, lfc_col)
    
    for idx in sig_up.index[:top_n_labels]:
        ax.annotate(
            idx,
            (de_results.loc[idx, lfc_col], de_results.loc[idx, 'neg_log10_pval']),
            fontsize=8,
            alpha=0.8,
        )
    
    for idx in sig_down.index[:top_n_labels]:
        ax.annotate(
            idx,
            (de_results.loc[idx, lfc_col], de_results.loc[idx, 'neg_log10_pval']),
            fontsize=8,
            alpha=0.8,
        )
    
    ax.set_xlabel('Log2 Fold Change')
    ax.set_ylabel('-log10(1 - P(DE))')
    ax.set_title(title)
    
    plt.tight_layout()
    return fig


# Plot volcano for global DE
if 'de_burn_sham' in dir():
    fig = plot_volcano(de_burn_sham, title='Burn vs Sham - All Cells')
    plt.savefig(ANALYSIS_DIR / 'volcano_burn_vs_sham.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. Temporal Analysis of Wound Healing

Analyze changes across timepoints to understand healing dynamics.

In [ ]:
# Check if timepoint data is available
if 'Timepoint' in adata.obs.columns:
    print("Timepoints available:")
    print(adata.obs['Timepoint'].value_counts())
    
    # Create Type_Timepoint column if not exists
    if 'Type_Timepoint' not in adata.obs.columns:
        adata.obs['Type_Timepoint'] = adata.obs['Type'].astype(str) + '_' + adata.obs['Timepoint'].astype(str)

In [ ]:
# UMAP colored by timepoint
if 'Timepoint' in adata.obs.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sc.pl.umap(adata, color='Timepoint', ax=axes[0], show=False, title='Timepoint')
    
    if 'Type_Timepoint' in adata.obs.columns:
        sc.pl.umap(adata, color='Type_Timepoint', ax=axes[1], show=False, title='Type + Timepoint')
    
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'umap_timepoint.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Cell type composition over time
def plot_temporal_composition(
    adata: ad.AnnData,
    cell_type_col: str = 'cell_type_harmonized',
    timepoint_col: str = 'Timepoint',
    condition_col: str = 'Type',
) -> plt.Figure:
    """
    Plot cell type composition changes over time for each condition.
    """
    if timepoint_col not in adata.obs.columns:
        print("No timepoint data available")
        return None
    
    # Filter to cells with timepoint data
    mask = adata.obs[timepoint_col].notna()
    adata_temporal = adata[mask].copy()
    
    if len(adata_temporal) == 0:
        return None
    
    conditions = adata_temporal.obs[condition_col].unique()
    n_conditions = len(conditions)
    
    fig, axes = plt.subplots(1, n_conditions, figsize=(6*n_conditions, 5))
    if n_conditions == 1:
        axes = [axes]
    
    for ax, condition in zip(axes, conditions):
        cond_mask = adata_temporal.obs[condition_col] == condition
        cond_data = adata_temporal[cond_mask]
        
        # Compute proportions
        ct_counts = pd.crosstab(cond_data.obs[timepoint_col], cond_data.obs[cell_type_col])
        ct_props = ct_counts.div(ct_counts.sum(axis=1), axis=0)
        
        # Sort timepoints
        ct_props = ct_props.sort_index()
        
        # Plot
        ct_props.plot(kind='bar', stacked=True, ax=ax, colormap='tab20', legend=False)
        ax.set_xlabel('Timepoint')
        ax.set_ylabel('Proportion')
        ax.set_title(f'{condition}')
        ax.tick_params(axis='x', rotation=45)
    
    # Add legend
    handles, labels = axes[-1].get_legend_handles_labels()
    fig.legend(handles, labels, loc='center right', bbox_to_anchor=(1.15, 0.5), title='Cell Type')
    
    plt.tight_layout()
    return fig


fig = plot_temporal_composition(adata)
if fig:
    plt.savefig(ANALYSIS_DIR / 'temporal_composition.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Temporal DE analysis - compare early vs late timepoints
if 'Timepoint' in adata.obs.columns:
    timepoints = sorted(adata.obs['Timepoint'].dropna().unique())
    
    if len(timepoints) >= 2:
        early = timepoints[0]
        late = timepoints[-1]
        
        print(f"Comparing {early} vs {late}")
        
        # For Burn wounds specifically
        burn_mask = adata.obs['Type'] == 'Burn'
        if burn_mask.sum() > 0:
            burn_adata = adata[burn_mask].copy()
            
            early_mask = burn_adata.obs['Timepoint'] == early
            late_mask = burn_adata.obs['Timepoint'] == late
            
            if early_mask.sum() >= 50 and late_mask.sum() >= 50:
                # Need to get indices in original adata
                early_idx = burn_adata.obs_names[early_mask]
                late_idx = burn_adata.obs_names[late_mask]
                
                early_mask_full = adata.obs_names.isin(early_idx)
                late_mask_full = adata.obs_names.isin(late_idx)
                
                de_temporal = model.differential_expression(
                    idx1=early_mask_full,
                    idx2=late_mask_full,
                )
                
                sig_temporal = de_temporal[
                    (de_temporal['is_de_fdr_0.05'] == True) &
                    (abs(de_temporal['lfc_mean']) > 0.5)
                ].sort_values('lfc_mean', ascending=False)
                
                print(f"\nSignificant temporal DE genes: {len(sig_temporal)}")
                print("\nUpregulated in late healing:")
                display(sig_temporal.head(10))
                print("\nDownregulated in late healing:")
                display(sig_temporal.tail(10))
                
                de_temporal.to_csv(ANALYSIS_DIR / f'de_burn_{early}_vs_{late}.csv')

## 6. Trajectory Analysis

Use diffusion pseudotime or PAGA to understand healing trajectories.

In [ ]:
# Compute diffusion map
sc.tl.diffmap(adata)
print("Diffusion map computed")

In [ ]:
# Visualize diffusion components
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sc.pl.diffmap(adata, color='cell_type_harmonized', ax=axes[0], show=False, components=['1,2'], title='Cell Type')
sc.pl.diffmap(adata, color='study', ax=axes[1], show=False, components=['1,2'], title='Study')

if 'Type' in adata.obs.columns:
    sc.pl.diffmap(adata, color='Type', ax=axes[2], show=False, components=['1,2'], title='Condition')

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'diffmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PAGA analysis for trajectory inference
sc.tl.paga(adata, groups='leiden_scvi')

fig, ax = plt.subplots(figsize=(10, 10))
sc.pl.paga(
    adata,
    color='leiden_scvi',
    ax=ax,
    show=False,
    threshold=0.1,
    node_size_scale=2,
)
plt.title('PAGA - Cluster Connectivity')
plt.savefig(ANALYSIS_DIR / 'paga_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PAGA for cell types
sc.tl.paga(adata, groups='cell_type_harmonized')

fig, ax = plt.subplots(figsize=(12, 12))
sc.pl.paga(
    adata,
    color='cell_type_harmonized',
    ax=ax,
    show=False,
    threshold=0.1,
    node_size_scale=2,
)
plt.title('PAGA - Cell Type Connectivity')
plt.savefig(ANALYSIS_DIR / 'paga_celltypes.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Wound Healing Signature Analysis

Define and score cells for wound healing-related gene signatures.

In [ ]:
# Define wound healing-related signatures
HEALING_SIGNATURES = {
    # Inflammation
    'Inflammatory_Response': ['Il1b', 'Il6', 'Tnf', 'Cxcl1', 'Cxcl2', 'Ccl2', 'Ccl3', 'Ccl4'],
    
    # Proliferation
    'Proliferation': ['Mki67', 'Top2a', 'Pcna', 'Mcm2', 'Cdk1', 'Ccnb1', 'Ccna2'],
    
    # ECM remodeling
    'ECM_Remodeling': ['Col1a1', 'Col3a1', 'Fn1', 'Mmp2', 'Mmp9', 'Mmp14', 'Timp1'],
    
    # Angiogenesis
    'Angiogenesis': ['Vegfa', 'Vegfc', 'Angpt1', 'Angpt2', 'Pecam1', 'Kdr', 'Flt1'],
    
    # Re-epithelialization
    'Re_epithelialization': ['Krt6a', 'Krt6b', 'Krt16', 'Krt17', 'Sprr1a', 'Sprr2a'],
    
    # Fibrosis
    'Fibrosis': ['Acta2', 'Col1a1', 'Tgfb1', 'Ctgf', 'Postn', 'Lox', 'Loxl2'],
    
    # Anti-inflammatory/Resolution
    'Resolution': ['Il10', 'Tgfb1', 'Arg1', 'Mrc1', 'Cd163', 'Retnla'],
    
    # Oxidative stress
    'Oxidative_Stress': ['Sod1', 'Sod2', 'Cat', 'Gpx1', 'Hmox1', 'Nfe2l2'],
    
    # Apoptosis
    'Apoptosis': ['Bax', 'Bcl2', 'Casp3', 'Casp9', 'Fas', 'Fasl'],
}

# Score cells for each signature
for sig_name, genes in HEALING_SIGNATURES.items():
    available_genes = [g for g in genes if g in adata.var_names]
    if len(available_genes) >= 2:
        sc.tl.score_genes(adata, gene_list=available_genes, score_name=f'sig_{sig_name}')
        print(f"{sig_name}: {len(available_genes)}/{len(genes)} genes available")
    else:
        print(f"{sig_name}: Insufficient genes ({len(available_genes)}/{len(genes)})")

In [ ]:
# Visualize signatures on UMAP
sig_cols = [c for c in adata.obs.columns if c.startswith('sig_')]

if sig_cols:
    n_cols = 3
    n_rows = int(np.ceil(len(sig_cols) / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows > 1 else [axes] if n_rows == 1 and n_cols == 1 else axes
    
    for ax, sig in zip(axes, sig_cols):
        sc.pl.umap(adata, color=sig, ax=ax, show=False, cmap='RdBu_r', title=sig.replace('sig_', ''))
    
    # Hide empty axes
    for ax in axes[len(sig_cols):]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'healing_signatures_umap.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# Compare signature scores across conditions
def compare_signatures_by_condition(
    adata: ad.AnnData,
    condition_col: str = 'Type',
) -> pd.DataFrame:
    """
    Compare signature scores across conditions.
    """
    sig_cols = [c for c in adata.obs.columns if c.startswith('sig_')]
    
    results = []
    conditions = adata.obs[condition_col].unique()
    
    for sig in sig_cols:
        row = {'Signature': sig.replace('sig_', '')}
        for cond in conditions:
            mask = adata.obs[condition_col] == cond
            row[f'mean_{cond}'] = adata.obs.loc[mask, sig].mean()
        results.append(row)
    
    return pd.DataFrame(results)


sig_comparison = compare_signatures_by_condition(adata)
display(sig_comparison)

In [ ]:
# Heatmap of signature scores by condition and cell type
def plot_signature_heatmap(
    adata: ad.AnnData,
    groupby: str = 'cell_type_harmonized',
) -> plt.Figure:
    """
    Plot heatmap of signature scores by cell type.
    """
    sig_cols = [c for c in adata.obs.columns if c.startswith('sig_')]
    
    # Calculate mean scores per group
    mean_scores = adata.obs.groupby(groupby)[sig_cols].mean()
    mean_scores.columns = [c.replace('sig_', '') for c in mean_scores.columns]
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(
        mean_scores.T,
        cmap='RdBu_r',
        center=0,
        ax=ax,
        cbar_kws={'label': 'Signature Score'},
    )
    ax.set_xlabel('Cell Type')
    ax.set_ylabel('Signature')
    ax.set_title('Wound Healing Signatures by Cell Type')
    plt.xticks(rotation=45, ha='right')
    
    plt.tight_layout()
    return fig


fig = plot_signature_heatmap(adata)
plt.savefig(ANALYSIS_DIR / 'signature_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Comparison Across Studies

Compare patterns across different experimental models.

In [ ]:
# Cell type composition by study
ct_study = pd.crosstab(
    adata.obs['cell_type_harmonized'],
    adata.obs['study'],
    normalize='columns'
)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(ct_study, cmap='YlOrRd', ax=ax, annot=True, fmt='.2f')
ax.set_xlabel('Study')
ax.set_ylabel('Cell Type')
ax.set_title('Cell Type Proportions by Study')

plt.tight_layout()
plt.savefig(ANALYSIS_DIR / 'celltype_by_study_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Find conserved markers across studies
def find_conserved_markers(
    adata: ad.AnnData,
    cell_type_col: str = 'cell_type_harmonized',
    study_col: str = 'study',
    n_genes: int = 100,
) -> pd.DataFrame:
    """
    Find marker genes that are consistent across studies.
    """
    studies = adata.obs[study_col].unique()
    cell_types = adata.obs[cell_type_col].unique()
    
    all_results = []
    
    for ct in cell_types:
        ct_markers = {}
        
        for study in studies:
            mask = (adata.obs[study_col] == study)
            if mask.sum() < 100:
                continue
                
            adata_study = adata[mask].copy()
            
            ct_mask = adata_study.obs[cell_type_col] == ct
            if ct_mask.sum() < 20:
                continue
            
            try:
                sc.tl.rank_genes_groups(
                    adata_study,
                    groupby=cell_type_col,
                    groups=[ct],
                    method='wilcoxon',
                    n_genes=n_genes,
                )
                
                markers = sc.get.rank_genes_groups_df(adata_study, group=ct)
                ct_markers[study] = set(markers['names'].head(n_genes))
            except:
                continue
        
        if len(ct_markers) >= 2:
            # Find intersection
            conserved = set.intersection(*ct_markers.values())
            all_results.append({
                'cell_type': ct,
                'n_studies': len(ct_markers),
                'n_conserved_markers': len(conserved),
                'conserved_markers': list(conserved)[:20],  # Top 20
            })
    
    return pd.DataFrame(all_results)


# This can take a while, so let's just show the concept
print("Computing conserved markers (this may take a few minutes)...")
conserved_markers = find_conserved_markers(adata, n_genes=50)
display(conserved_markers)

## 9. Pathological vs Normal Healing Analysis

Compare pathological conditions (Burn, KO models) to normal healing (Sham, WT).

In [ ]:
# Define pathological vs normal categories
pathological_types = ['Burn', 'CKO', 'LPCAT3 KO GAS']  # Modify based on your data
normal_types = ['Sham', 'WT', 'WT (NS)', 'LPCAT3 WT GAS']  # Modify based on your data

# Create healing category
def categorize_healing(type_val):
    if type_val in pathological_types:
        return 'Pathological'
    elif type_val in normal_types:
        return 'Normal'
    else:
        return 'Other'

adata.obs['healing_category'] = adata.obs['Type'].apply(categorize_healing)
print(adata.obs['healing_category'].value_counts())

In [ ]:
# Compare pathological vs normal
if 'Pathological' in adata.obs['healing_category'].values and 'Normal' in adata.obs['healing_category'].values:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    sc.pl.umap(adata, color='healing_category', ax=axes[0], show=False, title='Healing Category')
    
    # Signature comparison
    sig_cols = [c for c in adata.obs.columns if c.startswith('sig_')]
    if sig_cols:
        sig_data = adata.obs[sig_cols + ['healing_category']].melt(
            id_vars=['healing_category'],
            var_name='Signature',
            value_name='Score'
        )
        sig_data['Signature'] = sig_data['Signature'].str.replace('sig_', '')
        
        sig_data_filtered = sig_data[sig_data['healing_category'].isin(['Pathological', 'Normal'])]
        
        sns.boxplot(
            data=sig_data_filtered,
            x='Signature',
            y='Score',
            hue='healing_category',
            ax=axes[1],
        )
        axes[1].tick_params(axis='x', rotation=45)
        axes[1].set_title('Signature Scores: Pathological vs Normal')
    
    plt.tight_layout()
    plt.savefig(ANALYSIS_DIR / 'pathological_vs_normal.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# DE: Pathological vs Normal
if 'Pathological' in adata.obs['healing_category'].values and 'Normal' in adata.obs['healing_category'].values:
    path_mask = adata.obs['healing_category'] == 'Pathological'
    norm_mask = adata.obs['healing_category'] == 'Normal'
    
    print(f"Pathological: {path_mask.sum()} cells")
    print(f"Normal: {norm_mask.sum()} cells")
    
    de_path_vs_norm = model.differential_expression(
        idx1=norm_mask.values,
        idx2=path_mask.values,
    )
    
    sig_genes = de_path_vs_norm[
        (de_path_vs_norm['is_de_fdr_0.05'] == True) &
        (abs(de_path_vs_norm['lfc_mean']) > 0.5)
    ].sort_values('lfc_mean', ascending=False)
    
    print(f"\nSignificant genes: {len(sig_genes)}")
    print("\nUpregulated in Pathological:")
    display(sig_genes.head(15))
    print("\nDownregulated in Pathological:")
    display(sig_genes.tail(15))
    
    de_path_vs_norm.to_csv(ANALYSIS_DIR / 'de_pathological_vs_normal.csv')

## 10. Save Final Results

In [ ]:
# Save updated AnnData with all annotations
adata.write(OUTPUT_DIR / 'integrated_atlas_analyzed.h5ad')
print(f"Saved analyzed atlas to: {OUTPUT_DIR / 'integrated_atlas_analyzed.h5ad'}")

In [ ]:
# Export key results to CSV
# Cell type annotations
adata.obs[['study', 'Type', 'cell_type', 'cell_type_harmonized', 'leiden_scvi', 'batch']].to_csv(
    ANALYSIS_DIR / 'cell_annotations.csv'
)

# Signature scores
sig_cols = [c for c in adata.obs.columns if c.startswith('sig_')]
if sig_cols:
    adata.obs[['study', 'Type', 'cell_type_harmonized'] + sig_cols].to_csv(
        ANALYSIS_DIR / 'signature_scores.csv'
    )

# UMAP coordinates
umap_df = pd.DataFrame(
    adata.obsm['X_umap'],
    index=adata.obs_names,
    columns=['UMAP1', 'UMAP2']
)
umap_df.to_csv(ANALYSIS_DIR / 'umap_coordinates.csv')

print("Results exported to analysis directory")

## Summary

This analysis notebook provides:

1. **Cell type harmonization** - Unified cell type annotations across datasets
2. **Composition analysis** - Cell type proportions across conditions
3. **Differential expression** - Genes changed in burn/pathological conditions
4. **Temporal analysis** - Changes over healing timepoints
5. **Trajectory inference** - PAGA connectivity between cell types
6. **Signature scoring** - Wound healing-related pathway activity
7. **Cross-study comparison** - Conserved patterns across experimental models
8. **Pathological vs normal** - Key differences in healing phenotypes

All results are saved in the `output/analysis/` directory.